# Experiment: exp_001 soundscape reference blend

Objective:
- Rebuild the reference inference stack inside the project notebook workflow.
- Establish a trustworthy local baseline on labeled soundscapes before starting new training.
- Compare four strategies: `LB862`, `LB872`, `0.8 * LB872 + 0.2 * LB862`, and the same blend with notebook heuristics.

Success criteria:
- Verify local labeled soundscape coverage and row construction.
- Produce a local macro ROC-AUC comparison table when the notebook runs in a Torch-enabled kernel.
- Save notebook-driven outputs under `experiments/outputs/exp_001_soundscape_reference_blend/`.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from birdclef2026.reference_eval import (
    DEFAULT_STRATEGIES,
    build_dry_run_payload,
    build_validation_targets,
    metrics_to_summary_frame,
    run_reference_evaluation,
)

DATA_DIR = PROJECT_ROOT / "data" / "birdclef-2026"
MODEL_DIR = PROJECT_ROOT / "data" / "BirdCLEF-2026-model"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "outputs" / "exp_001_soundscape_reference_blend"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

{
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "model_dir": str(MODEL_DIR),
    "output_dir": str(OUTPUT_DIR),
}


## Plan

- Hypothesis: the reference blend is a stronger starting point than either checkpoint alone, but the leaderboard heuristics still need local validation.
- Strategies to compare:
  - `baseline_only`
  - `finetuned_only`
  - `prob_ft80_base20`
  - `prob_ft80_base20_plus_heuristics`
- Metrics to record:
  - local macro ROC-AUC
  - number of scored classes
  - classes skipped because local positives are missing
  - notes on rare and soundscape-only species coverage


In [ ]:
targets = build_validation_targets(DATA_DIR)
dry_run_payload = build_dry_run_payload(targets, DATA_DIR, MODEL_DIR)

with open(OUTPUT_DIR / "dry_run_summary.json", "w", encoding="utf-8") as handle:
    json.dump(dry_run_payload, handle, indent=2, sort_keys=True)

display(pd.Series(targets.summary, name="value").to_frame())
dry_run_payload["first_row_ids"][:10]


In [ ]:
local_positive_counts = targets.label_df.sum(axis=0).sort_values(ascending=False)
coverage_table = pd.DataFrame(
    {"local_positive_segments": local_positive_counts}
)
coverage_table = coverage_table[coverage_table["local_positive_segments"] > 0]

display(coverage_table.head(15))
coverage_table.tail(15)


## Experiment Configuration

- Use `RUN_FULL_EVAL = False` for quick notebook inspection without Torch.
- Switch it to `True` only inside a kernel that has `torch`, `timm`, `torchaudio`, `librosa`, and `scikit-learn`.
- The notebook writes outputs into the experiment-specific output directory so results can be reloaded later.


In [ ]:
RUN_FULL_EVAL = False
REQUESTED_STRATEGIES = [strategy.name for strategy in DEFAULT_STRATEGIES]
LIMIT_FILES = None
SAVE_PREDICTIONS = False
DEVICE = "auto"

experiment_config = {
    "run_full_eval": RUN_FULL_EVAL,
    "requested_strategies": REQUESTED_STRATEGIES,
    "limit_files": LIMIT_FILES,
    "save_predictions": SAVE_PREDICTIONS,
    "device": DEVICE,
}
experiment_config


In [ ]:
metrics_result = None

if RUN_FULL_EVAL:
    metrics_result = run_reference_evaluation(
        data_dir=DATA_DIR,
        model_dir=MODEL_DIR,
        output_dir=OUTPUT_DIR,
        device_name=DEVICE,
        requested_strategies=REQUESTED_STRATEGIES,
        limit_files=LIMIT_FILES,
        save_predictions=SAVE_PREDICTIONS,
    )
    print("Full evaluation finished.")
else:
    print("Full evaluation skipped. Set RUN_FULL_EVAL = True in a Torch-enabled kernel.")


In [ ]:
metrics_path = OUTPUT_DIR / "metrics.json"

if metrics_result is None and metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as handle:
        metrics_result = json.load(handle)

if metrics_result is None:
    print("No metrics available yet.")
else:
    summary_frame = metrics_to_summary_frame(metrics_result["metrics_by_strategy"])
    display(summary_frame)
    best_strategy = summary_frame.iloc[0].to_dict()
    best_strategy


## Results and Notes

- Current notebook status: dry-run coverage is validated.
- Full model evaluation is intentionally deferred until this notebook runs in a Torch-enabled kernel.
- After the first full run, copy the best strategy and macro ROC-AUC into `MASTER_EXPERIMENT_TABLE.md` and `PROJECT_STATE.md`.


In [ ]:
result_snapshot = {
    "experiment_id": "exp_001",
    "status": "pending_full_eval" if metrics_result is None else "completed",
    "targets_summary": targets.summary,
    "available_metrics": None
    if metrics_result is None
    else metrics_to_summary_frame(metrics_result["metrics_by_strategy"]).to_dict(orient="records"),
}

with open(OUTPUT_DIR / "notebook_result_snapshot.json", "w", encoding="utf-8") as handle:
    json.dump(result_snapshot, handle, indent=2, sort_keys=True)

result_snapshot


## Next Steps

- Run the notebook in a Torch-enabled kernel and persist `metrics.json`.
- Update the experiment registry with the winning strategy.
- Use the best validated reference setup as the starting point for the first training notebook.
